# INF2006 pipeline: fresh-session rerun on EMR

One section per stage, each calling the owned scripts in this repository (no logic lives in the notebook).
Run on a **fresh** EMR cluster that starts from the raw files in S3 only. Every section ends with verification
checks against the committed reference evidence: check, command, expected, actual, evidence location, result.

How to run: `docs/rerun/appendix_a_rerun_guide.md` (Appendix A). Parameters come from environment variables:
`RERUN_ID` (new run id), `S3_ROOT` (`s3://<bucket>/inf2006`), `WITH_GDELT` (1 = full run with the GDELT parse, about 1 hour).

The 2017 test set is **not** evaluated here: it was evaluated once (Task 3, Gate 2). This notebook reloads the saved
model and checks that it reproduces the stored test predictions without retraining.

In [ ]:
import os

from rerun_lib import Rerun, get, max_auc_diff, problems_total, q1_paired, q2_paired, q3a_dispersion, repo_root

REPO = repo_root()
os.chdir(REPO)
RERUN_ID = os.environ["RERUN_ID"]
S3_ROOT = os.environ["S3_ROOT"].rstrip("/")
WITH_GDELT = os.environ.get("WITH_GDELT", "1") == "1"
DRY = os.environ.get("DRY_RUN") == "1"

# Reference runs whose committed evidence this rerun must reproduce
REF_T2, REF_MODEL, REF_T4 = "20261105-1430-full-ref", "20261106-1400-full-ref", "20261110-1000-full-ref"
FROZEN = "lr_f1_r001"
PRICES = "/inf2006/raw/prices/plotly-datasets-0c447c4/acquired=2026-11-03"

rr = Rerun(RERUN_ID, dry=DRY)
print("repo", REPO, "| rerun", RERUN_ID, "| with GDELT", WITH_GDELT, "| dry run", DRY)

## 0. Environment record (software versions, cluster, code version)

In [ ]:
rr.sh(
    "date -u; git rev-parse HEAD; git status --short | head -5; python3 --version; hadoop version | head -1; "
    "spark-submit --version 2>&1 | grep -m1 version; hive --version 2>/dev/null | head -1; "
    "grep -io 'emr-[0-9.]*' /mnt/var/lib/info/extraInstanceData.json | head -1; "
    "yarn node -list 2>/dev/null | tail -n +2",
    "environment",
)

## 1. Task 1: stage the raw corpus from S3 into HDFS and verify it
All four sources (price CSV, GDELT, SEC EDGAR, Wikinews) are copied unchanged and checked against the committed manifests.

In [ ]:
rr.sh(f"bash corpus/stage_to_hdfs.sh {S3_ROOT}/raw {RERUN_ID}", "task1_stage_and_verify")
inv = rr.load(f"evidence/corpus/inventory_{RERUN_ID}.json") or {}
rr.check(
    "HDFS copies match their manifests (files)",
    "corpus/stage_to_hdfs.sh",
    1047,
    inv.get("files_ok"),
    f"evidence/corpus/inventory_{RERUN_ID}.json",
)
rr.check(
    "Verification problems across all sources",
    "corpus/verify_hdfs.py",
    0,
    problems_total(inv),
    f"evidence/corpus/hdfs_verify_{RERUN_ID}.csv",
)

## 2. Task 2: data pipeline (Streaming audit, PySpark cleaning and features, Hive tables)
`WITH_GDELT=1` also runs the GDELT Streaming job over all 731 archives (about 52 minutes).

In [ ]:
rr.sh(f"WITH_GDELT={int(WITH_GDELT)} bash spark/run_task2.sh {RERUN_ID} {PRICES}", "task2_pipeline")
ref = rr.load(f"evidence/pipeline/reconciliation_{REF_T2}.json") or {}
new = rr.load(f"evidence/pipeline/reconciliation_{RERUN_ID}.json") or {}
ev = f"evidence/pipeline/reconciliation_{RERUN_ID}.json"
rr.check("Raw = clean + rejected price rows", "spark/clean_prices.py", True, get(new, "prices", "reconciles"), ev)
rr.check("Clean price rows", "spark/clean_prices.py", 605887, get(new, "prices", "clean_rows"), ev)
for split in ("train", "validation", "test"):
    rr.check(
        f"Eligible stock-weeks, {split}",
        "spark/weekly_features.py",
        get(ref, "features_weekly", split, "eligible"),
        get(new, "features_weekly", split, "eligible"),
        ev,
    )
if WITH_GDELT:
    rr.check(
        "GDELT events read (731 archives)", "streaming/gdelt_daily/run.sh", 144207180, get(new, "gdelt", "lines"), ev
    )
    rr.check("GDELT corrupt archives", "streaming/gdelt_daily/run.sh", 0, get(new, "gdelt", "corrupt_archives"), ev)
    rr.check(
        "Weeks with a pre-start news measure",
        "hive/41_weekly_news.sql",
        97,
        get(new, "weekly_news_weeks_with_news_abn"),
        ev,
    )

## 3a. Task 3: rerun model selection on 2016 validation (2017 is not read)
Refits the eight Gate 1 configurations on 2013 to 2015 with seed 2026 and checks that the same configuration is recommended.

In [ ]:
rr.sh(f"bash modeling/run_task3.sh validate {RERUN_ID}", "task3_validation")
vref = rr.load(f"evidence/model/validation_{REF_MODEL}.json") or {}
vnew = rr.load(f"evidence/model/validation_{RERUN_ID}.json") or {}
ev = f"evidence/model/validation_{RERUN_ID}.json"
rr.check(
    "Recommended frozen configuration",
    "modeling/run_model.py validate",
    FROZEN,
    get(vnew, "selection", "recommended_frozen_config"),
    ev,
)
rr.check("Test split read during validation", "modeling/run_model.py validate", False, vnew.get("test_split_read"), ev)
rr.check(
    "Max difference in mean weekly AUC, 8 configs vs reference run",
    "modeling/run_model.py validate",
    0.0,
    max_auc_diff(vref, vnew),
    ev,
    tol=1e-6,
)

## 3b. Task 3: reload the saved model and reproduce the stored test predictions without retraining
The saved PipelineModel comes from this repository (`models/`), the stored predictions and weekly evaluation from S3.

In [ ]:
rr.sh(
    f"hdfs dfs -mkdir -p /inf2006/models/{REF_MODEL} && "
    f"hdfs dfs -put -f models/{REF_MODEL}/pipeline_model /inf2006/models/{REF_MODEL}/ && "
    f"s3-dist-cp --src {S3_ROOT}/predictions/{REF_MODEL} --dest hdfs:///inf2006/predictions/{REF_MODEL} && "
    f"s3-dist-cp --src {S3_ROOT}/eval/{REF_MODEL} --dest hdfs:///inf2006/eval/{REF_MODEL}",
    "task3_copy_model_and_predictions",
)
rr.sh(f"spark-submit modeling/run_model.py reload --run-id {REF_MODEL} --root /inf2006 --out {rr.out}", "task3_reload")
rl = rr.load(f"{rr.out}/reload_{REF_MODEL}.json") or {}
ev = f"{rr.out}/reload_{REF_MODEL}.json"
rr.check("Reload check passed (no retraining)", "modeling/run_model.py reload", True, rl.get("passed"), ev)
rr.check("Test predictions compared", "modeling/run_model.py reload", 25531, rl.get("rows_compared"), ev)
rr.check(
    "Max absolute prediction difference", "modeling/run_model.py reload", 0.0, rl.get("max_abs_diff"), ev, tol=1e-9
)

## 4. Task 4: analytics (Hive Q1 and Q3a, PySpark Q2 and Q3b) on this session's features and the stored predictions

In [ ]:
rr.sh(f"bash analytics/run_task4.sh {RERUN_ID} /inf2006 {RERUN_ID} {REF_MODEL}", "task4_analytics")
aref = rr.load(f"evidence/analytics/task4_{REF_T4}.json") or {}
anew = rr.load(f"evidence/analytics/task4_{RERUN_ID}.json") or {}
ev = f"evidence/analytics/task4_{RERUN_ID}.json"
rr.check(
    "Q2 paired AUC difference, 2017, all weeks",
    "analytics/run_analytics.py",
    q2_paired(aref),
    q2_paired(anew),
    ev,
    tol=1e-9,
)
rr.check(
    "Q1 busiest minus quietest quintile, 2017",
    "analytics/20_q1_volume_quintiles.sql",
    q1_paired(aref),
    q1_paired(anew),
    ev,
    tol=1e-9,
)
if WITH_GDELT:
    rr.check(
        "Q3a dispersion, high minus low news tercile",
        "analytics/30_q3a_news_market.sql",
        q3a_dispersion(aref),
        q3a_dispersion(anew),
        ev,
        tol=1e-9,
    )

## 5. Verification summary (Appendix A record) and stage times

In [ ]:
summary = rr.report()
print(f"\nSaved: {rr.out}/rerun_{RERUN_ID}.json, {rr.out}/checks_{RERUN_ID}.csv, {rr.out}/timings_{RERUN_ID}.csv")